# Evaluate AI-Q research traces with MLflow

This notebook evaluates **completed AI-Q research executions** from traces already stored in MLflow. It does not submit a new research request or rerun the application. The main path uses deterministic, code-based scorers to describe how the workflow executed: latency, model activity, tool use, parallelism, errors, token data, and trace structure.

## What you will do

1. Connect an OpenShift AI workbench to the MLflow tracking server used by AI-Q.
2. Select a reusable evaluation set of completed AI-Q traces.
3. Resolve each dataset record to its complete historical trace.
4. Inspect the spans AI-Q emitted for models, tools, searches, delegated work, and file operations.
5. Run deterministic operational scorers against those traces.
6. Review the resulting `genai_evaluate` run and per-trace assessments in MLflow.
7. Optionally add explicit expectation checks after you have meaningful baselines or requirements.

## Before you start

You should have:

- an AI-Q deployment that has already completed at least one research workflow
- AI-Q traces visible in an MLflow experiment
- an OpenShift AI workbench that can reach the MLflow tracking service
- permission for the workbench service account to read traces, datasets, and create evaluation runs

### Recommended: create an evaluation dataset in MLflow

Using a dataset makes the evaluation set stable and reusable instead of depending on whichever traces happen to match a search at run time.

1. Open the MLflow experiment that contains the AI-Q traces.
2. Open **Traces** and select one or more completed research traces that represent the scenarios you want to evaluate.
3. Choose **Actions → Add to evaluation dataset**.
4. Create a new dataset or add the traces to an existing dataset.
5. In this notebook, set `TRACE_EXPERIMENT_ID` to the source experiment.
6. Set `DATASET_ID` to the dataset you want to evaluate. Section 4 also lists datasets associated with the experiment if you need to confirm the ID.

`TRACE_IDS` and `TRACE_FILTER` are available as fallback selection methods for development or troubleshooting, but `DATASET_ID` is the recommended quickstart path.

## What the notebook creates

The original AI-Q traces remain intact. MLflow creates a **new evaluation run in the same experiment** and associates the evaluated traces and their assessments with that run. The notebook does not create a separate evaluation experiment.

## 1. Prepare the workbench

Run the installation cell once for the workbench image. Restart the kernel only if MLflow was already imported before the installation completed.

> **Live demo:** Skip the installation cell when the environment is already prepared.


In [ ]:
# Run once per workbench environment, then restart the kernel if needed.
%pip install -q --extra-index-url https://pypi.org/simple/ \
    "mlflow[kubernetes]>=3.11,<4" \
    "kubernetes>=30.0.0" \
    "pandas>=2.2.0" \
    "pyarrow>=15.0.0" \
    "requests>=2.32.0"


## 2. Choose the MLflow connection and trace source

This is the main configuration cell.

### Trace selection priority

1. `DATASET_ID` — recommended; evaluates a stable dataset created from selected MLflow traces
2. `TRACE_IDS` — evaluates explicitly supplied trace IDs when no dataset is configured
3. `TRACE_FILTER` — searches the source experiment and evaluates the matching traces when neither of the above is configured

The quickstart is designed around `DATASET_ID` because the selected traces stay visible and reusable as an evaluation set in MLflow.

### Experiment placement

`TRACE_EXPERIMENT_ID` identifies the experiment containing the original AI-Q traces. The notebook writes the evaluation run back to that same experiment so the source executions, evaluation run, and assessments stay together.

### Optional expectation checks

Leave `ENABLE_EXPECTATION_CHECKS=False` for the main operational evaluation. Section 10 explains how to enable deterministic pass/fail checks after you have established scenario-specific requirements or baselines.


In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------
# Red Hat OpenShift AI MLflow connection
# ---------------------------------------------------------------------
MLFLOW_TRACKING_URI = (
    "https://mlflow.redhat-ods-applications."
    "svc.cluster.local:8443"
)

# Matches the supplied working example. If authentication fails in a
# newer environment, try: kubernetes-namespaced
MLFLOW_TRACKING_AUTH = "kubernetes"

# TLS modes:
# - "insecure": skip certificate verification; appropriate for this
#   LaunchPad quickstart and matches the supplied working notebook.
# - "service-ca": use an OpenShift service CA for an in-cluster .svc URI.
# - "system": use the workbench image's normal system trust bundle.
MLFLOW_TLS_MODE = "insecure"

# Leave blank to auto-detect the workbench namespace.
MLFLOW_WORKSPACE = ""

# Experiment where AI-Q's native traces are stored.
TRACE_EXPERIMENT_ID = 1

# ---------------------------------------------------------------------
# Trace source selection
# Priority: DATASET_ID -> TRACE_IDS -> TRACE_FILTER
# ---------------------------------------------------------------------
DATASET_ID = ""

TRACE_IDS = [
    # "tr-REPLACE_WITH_TRACE_ID",
]

TRACE_FILTER = "trace.status = 'OK'"
MAX_TRACES = 10

# ---------------------------------------------------------------------
# Notebook behavior
# ---------------------------------------------------------------------
RUN_OPERATIONAL_EVALUATION = True

# Keep False for the main quickstart path.
# Section 10 explains how and when to enable this.
ENABLE_EXPECTATION_CHECKS = False

EXPORT_NAME = "aiq_mlflow_trace_export"

# Full traces may contain prompts, retrieved data, and responses.
INCLUDE_TRACE_CONTENT_IN_EXPORT = True

# Use span-name inference only when standard MLflow span types are absent.
ALLOW_SPAN_NAME_FALLBACK = False

print("Configuration loaded.")
print("  Tracking URI:", MLFLOW_TRACKING_URI)
print("  Tracking auth:", MLFLOW_TRACKING_AUTH)
print("  TLS mode:", MLFLOW_TLS_MODE)
print("  Workspace override:", MLFLOW_WORKSPACE or "<auto-detect>")
print("  Trace experiment ID:", TRACE_EXPERIMENT_ID)
print("  Dataset ID:", DATASET_ID or "<not set>")
print("  Explicit trace IDs:", len(TRACE_IDS))
print("  Search filter:", TRACE_FILTER)
print("  Expectation checks enabled:", ENABLE_EXPECTATION_CHECKS)


## 3. Connect to MLflow and verify the context

This cell authenticates with the workbench service account, determines the namespace used as the MLflow workspace, and verifies that the source experiment is accessible.

The connection is ready when the output shows the expected:

- internal MLflow tracking URI
- workbench namespace
- `rh-research` experiment and experiment ID

> **Demo cue:** This establishes that the notebook is using the same MLflow environment where AI-Q recorded the research trace.


In [ ]:
import json
import logging
import re
import sys
import warnings
from typing import Any, Iterable, Optional

import mlflow
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("mlflow").setLevel(logging.ERROR)

from urllib3.exceptions import InsecureRequestWarning

warnings.filterwarnings(
    "ignore",
    category=InsecureRequestWarning,
)

version_parts = tuple(
    int(part)
    for part in mlflow.__version__.split(".")[:2]
)
if version_parts < (3, 11):
    raise RuntimeError(
        f"MLflow {mlflow.__version__} is installed. "
        "This notebook requires MLflow 3.11 or later. "
        "Run the install cell, restart the kernel, and rerun."
    )

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_AUTH"] = MLFLOW_TRACKING_AUTH

if not MLFLOW_WORKSPACE:
    namespace_candidates = [
        Path(
            "/run/secrets/kubernetes.io/"
            "serviceaccount/namespace"
        ),
        Path(
            "/var/run/secrets/kubernetes.io/"
            "serviceaccount/namespace"
        ),
    ]

    for namespace_path in namespace_candidates:
        if namespace_path.is_file():
            MLFLOW_WORKSPACE = (
                namespace_path.read_text().strip()
            )
            break

if not MLFLOW_WORKSPACE:
    raise RuntimeError(
        "Could not determine the MLflow workspace namespace."
    )

os.environ["MLFLOW_WORKSPACE"] = MLFLOW_WORKSPACE


# Configure TLS explicitly. Do not combine MLFLOW_TRACKING_INSECURE_TLS
# with MLFLOW_TRACKING_SERVER_CERT_PATH.
def configure_mlflow_tls(mode: str) -> None:
    mode = mode.lower().strip()

    conflicting_variables = [
        "MLFLOW_TRACKING_INSECURE_TLS",
        "MLFLOW_TRACKING_SERVER_CERT_PATH",
        "REQUESTS_CA_BUNDLE",
        "CURL_CA_BUNDLE",
    ]

    for variable in conflicting_variables:
        os.environ.pop(variable, None)

    if mode == "insecure":
        # Quickstart/development behavior for the supplied environment.
        os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
        print(
            "TLS mode: insecure. Certificate verification is disabled "
            "for the MLflow tracking connection."
        )
        return

    if mode == "service-ca":
        # Use only with an in-cluster MLflow service URI whose
        # certificate was issued by the OpenShift service CA.
        service_ca_candidates = [
            Path("/etc/ssl/certs/openshift-service-ca.crt"),
            Path(
                "/var/run/secrets/kubernetes.io/"
                "serviceaccount/service-ca.crt"
            ),
        ]
        service_ca = next(
            (
                path
                for path in service_ca_candidates
                if path.is_file()
            ),
            None,
        )

        if service_ca is None:
            raise RuntimeError(
                "MLFLOW_TLS_MODE='service-ca' was selected, but no "
                "OpenShift service CA file was found."
            )

        os.environ["MLFLOW_TRACKING_SERVER_CERT_PATH"] = str(
            service_ca
        )
        print("TLS mode: OpenShift service CA:", service_ca)
        return

    if mode == "system":
        print("TLS mode: system trust bundle.")
        return

    raise ValueError(
        "MLFLOW_TLS_MODE must be one of: "
        "'insecure', 'service-ca', or 'system'."
    )


configure_mlflow_tls(MLFLOW_TLS_MODE)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_workspace(MLFLOW_WORKSPACE)

print("Tracking URI being used:", MLFLOW_TRACKING_URI)
print("Auth mode:", os.environ["MLFLOW_TRACKING_AUTH"])
print("Workspace:", MLFLOW_WORKSPACE)

assert ".svc.cluster.local" in MLFLOW_TRACKING_URI, (
    "The tracking URI is not the internal MLflow service. "
    f"Current value: {MLFLOW_TRACKING_URI}"
)

source_experiment = mlflow.get_experiment(TRACE_EXPERIMENT_ID)
if source_experiment is None:
    raise RuntimeError(
        f"Trace experiment {TRACE_EXPERIMENT_ID!r} was not found."
    )

# Keep evaluation runs in the source trace experiment.
active_experiment = mlflow.set_experiment(
    experiment_id=str(TRACE_EXPERIMENT_ID)
)

print("\nConnected successfully:")
print("  MLflow version:", mlflow.__version__)
print("  Tracking URI:", mlflow.get_tracking_uri())
print("  Tracking auth:", os.environ.get("MLFLOW_TRACKING_AUTH"))
print("  TLS mode:", MLFLOW_TLS_MODE)
print("  Workspace:", MLFLOW_WORKSPACE)
print(
    "  Active experiment (source traces + evaluation runs):",
    active_experiment.name,
    f"({active_experiment.experiment_id})",
)


## 4. Locate the evaluation dataset

A dataset created in the MLflow UI provides a stable selection of historical traces to evaluate. This cell lists datasets associated with the source experiment.

A trace-backed record generally stores a source reference similar to:

```python
{"trace": {"trace_id": "tr-..."}}
```

The dataset holds the selection and optional record-level expectations. The full spans are retrieved from MLflow later.

> **Demo cue:** Evaluation starts from a governed, reusable dataset rather than an ad hoc trace search.


In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

datasets = client.search_datasets(
    experiment_ids=[str(TRACE_EXPERIMENT_ID)],
    filter_string="created_time >= 0",
    max_results=50,
    order_by=["last_update_time DESC"],
    page_token="",   # important workaround
)

dataset_rows = []
for dataset_item in datasets:
    dataset_rows.append(
        {
            "dataset_id": dataset_item.dataset_id,
            "name": dataset_item.name,
            "experiment_ids": getattr(
                dataset_item,
                "experiment_ids",
                None,
            ),
            "created_by": getattr(
                dataset_item,
                "created_by",
                None,
            ),
            "created_time": getattr(
                dataset_item,
                "created_time",
                None,
            ),
            "last_update_time": getattr(
                dataset_item,
                "last_update_time",
                None,
            ),
            "tags": getattr(dataset_item, "tags", None),
        }
    )

datasets_df = pd.DataFrame(dataset_rows)

if datasets_df.empty:
    print("No datasets were returned for this experiment.")
else:
    display(datasets_df)

### Confirm the configured dataset

This cell loads `DATASET_ID` directly and confirms its name, associated experiments, record count, and columns.

> **Checkpoint:** The dataset should exist in the same workspace and contain at least one trace-backed record.


In [ ]:
from mlflow.genai.datasets import (
    get_dataset,
    search_datasets,
)

# Directly verify the dataset whose ID you copied from the UI.
if DATASET_ID:
    try:
        selected_dataset = get_dataset(
            dataset_id=DATASET_ID
        )

        print("Dataset found directly:")
        print("  ID:", selected_dataset.dataset_id)
        print("  Name:", selected_dataset.name)
        print(
            "  Experiment IDs:",
            selected_dataset.experiment_ids,
        )

        selected_dataset_df = selected_dataset.to_df()

        print("  Records:", len(selected_dataset_df))
        print(
            "  Columns:",
            list(selected_dataset_df.columns),
        )

        display(selected_dataset_df.head())

    except Exception as exc:
        print("Could not load DATASET_ID directly.")
        print(type(exc).__name__, str(exc))

## 5. Preview the source traces

This optional view shows recent traces in the source experiment, including status, duration, request, response, tags, and metadata.

It provides context, but it does not change the evaluation set when `DATASET_ID` is configured.


In [ ]:
recent_traces_df = mlflow.search_traces(
    locations=[str(TRACE_EXPERIMENT_ID)],
    filter_string=TRACE_FILTER or None,
    order_by=["timestamp_ms DESC"],
    max_results=max(MAX_TRACES, 20),
)

print("Returned columns:")
print(list(recent_traces_df.columns))

preferred_columns = [
    "trace_id",
    "request_id",
    "trace_name",
    "name",
    "request_time",
    "timestamp_ms",
    "state",
    "status",
    "execution_duration",
    "execution_time_ms",
    "inputs",
    "request",
    "outputs",
    "response",
    "tags",
    "trace_metadata",
    "request_metadata",
]

visible_columns = [
    column
    for column in preferred_columns
    if column in recent_traces_df.columns
]

if recent_traces_df.empty:
    print("No traces matched the current filter.")
else:
    display(recent_traces_df[visible_columns].head(20))

## 6. Resolve dataset records into complete traces

The dataset stores trace references. This step follows those references and retrieves each complete MLflow `Trace` object, including its nested spans.

The resulting DataFrame is the evaluation input. Each row contains:

- the historical trace
- the trace ID
- optional dataset expectations
- dataset record metadata

> **Checkpoint:** The `trace` column must contain real MLflow trace objects. This lets the scorers inspect the execution rather than only the top-level request and response.


In [ ]:
def recursively_find_trace_id(value: Any) -> Optional[str]:
    """Find the first MLflow trace ID in a nested dataset source value."""
    if value is None:
        return None

    direct_trace_id = getattr(value, "trace_id", None)
    if isinstance(direct_trace_id, str) and direct_trace_id.startswith("tr-"):
        return direct_trace_id

    for converter_name in ("to_dict", "model_dump", "dict"):
        converter = getattr(value, converter_name, None)
        if callable(converter):
            try:
                converted = converter()
                if converted is not value:
                    result = recursively_find_trace_id(converted)
                    if result:
                        return result
            except Exception:
                pass

    if isinstance(value, str):
        if value.startswith("tr-"):
            return value

        # Handle serialized JSON or a trace ID embedded in a source string.
        stripped = value.strip()
        if stripped.startswith("{") or stripped.startswith("["):
            try:
                result = recursively_find_trace_id(json.loads(stripped))
                if result:
                    return result
            except Exception:
                pass

        match = re.search(r"\btr-[A-Za-z0-9_-]+\b", value)
        return match.group(0) if match else None

    if isinstance(value, dict):
        for candidate_key in (
            "trace_id",
            "source_trace_id",
            "request_id",
        ):
            candidate = value.get(candidate_key)
            if isinstance(candidate, str) and candidate.startswith("tr-"):
                return candidate

        for child in value.values():
            result = recursively_find_trace_id(child)
            if result:
                return result

    if isinstance(value, (list, tuple, set)):
        for child in value:
            result = recursively_find_trace_id(child)
            if result:
                return result

    return None

def get_trace_safely(trace_id: str):
    trace = mlflow.get_trace(trace_id)

    if trace is None:
        raise RuntimeError(
            f"Trace {trace_id!r} could not be retrieved."
        )

    return trace


dataset = None
dataset_df = None
trace_rows = []

if DATASET_ID:
    print("Loading traces from evaluation dataset:", DATASET_ID)
    dataset = get_dataset(dataset_id=DATASET_ID)
    dataset_df = dataset.to_df()

    print("Dataset name:", dataset.name)
    print("Dataset records:", len(dataset_df))
    print("Dataset columns:", list(dataset_df.columns))
    display(dataset_df.head())

    missing_source_records = []

    for row_index, record in dataset_df.iterrows():
        trace_id = recursively_find_trace_id(
            record.get("source")
        )

        if not trace_id:
            missing_source_records.append(row_index)
            continue

        trace_rows.append(
            {
                "trace_id": trace_id,
                "trace": get_trace_safely(trace_id),
                "inputs": record.get("inputs"),
                "outputs": record.get("outputs"),
                "expectations": (
                    record.get("expectations")
                    if isinstance(
                        record.get("expectations"),
                        dict,
                    )
                    else {}
                ),
                "dataset_record_id": record.get(
                    "dataset_record_id"
                ),
                "dataset_source": record.get("source"),
            }
        )

    if missing_source_records:
        print(
            "\nDataset records without a source trace were skipped:",
            missing_source_records,
        )

elif TRACE_IDS:
    print("Loading explicitly selected trace IDs.")
    for trace_id in TRACE_IDS:
        trace_rows.append(
            {
                "trace_id": trace_id,
                "trace": get_trace_safely(trace_id),
                "expectations": {},
            }
        )

else:
    print(
        "No dataset ID or explicit trace IDs were set. "
        "Using trace search."
    )

    searched_df = mlflow.search_traces(
        locations=[TRACE_EXPERIMENT_ID],
        filter_string=TRACE_FILTER or None,
        order_by=["timestamp_ms DESC"],
        max_results=MAX_TRACES,
    )

    if searched_df.empty:
        raise RuntimeError(
            "No traces matched the current search settings."
        )

    for _, row in searched_df.iterrows():
        trace = row.get("trace") if "trace" in searched_df.columns else None
        trace_id = (
            getattr(getattr(trace, "info", None), "trace_id", None)
            or row.get("trace_id")
            or row.get("request_id")
        )

        if trace is None and trace_id:
            trace = get_trace_safely(str(trace_id))

        if trace is None:
            print("Skipping a search result without a retrievable Trace object.")
            continue

        trace_rows.append(
            {
                "trace_id": trace_id,
                "trace": trace,
                "expectations": {},
            }
        )

trace_eval_df = pd.DataFrame(trace_rows)

if trace_eval_df.empty:
    raise RuntimeError(
        "No source traces were loaded. Confirm that the dataset "
        "was created from traces or select trace IDs directly."
    )

print("\nLoaded source traces:", len(trace_eval_df))
display(
    trace_eval_df[
        [
            column
            for column in [
                "trace_id",
                "dataset_record_id",
                "expectations",
            ]
            if column in trace_eval_df.columns
        ]
    ]
)

# Pass only MLflow's recognized evaluation columns into evaluate().
evaluation_columns = ["trace"]
if "expectations" in trace_eval_df.columns:
    evaluation_columns.append("expectations")

evaluation_data_df = trace_eval_df[evaluation_columns].copy()


## 7. Inspect the trace data emitted by AI-Q

Before calculating operational metrics, this section looks at the spans and metadata AI-Q recorded in MLflow. This helps identify which model calls, tool calls, timing data, and other execution details are available for evaluation.

The current workflow includes:

- `LLM` spans for Nemotron and GPT-OSS calls
- `TOOL` spans for search, file, planning, source-validation, and delegated-task operations
- `CHAIN` spans for workflow and agent-like orchestration
- nested wrapper and leaf operations, such as `advanced_web_search_tool` around `tavily_search`
- duration, status, parent-child, model, and token-related attributes

> **Demo cue:** This inventory connects observability data to the evaluation logic that follows.


In [ ]:
def compact_json(
    value: Any,
    limit: int = 700,
) -> str:
    try:
        rendered = json.dumps(
            value,
            ensure_ascii=False,
            default=str,
            sort_keys=True,
        )
    except Exception:
        rendered = str(value)

    if len(rendered) > limit:
        return rendered[:limit] + "...<truncated>"

    return rendered


def enum_text(value: Any) -> str:
    if value is None:
        return ""

    raw = getattr(value, "value", value)
    return str(raw)


def span_type_text(span: Any) -> str:
    return enum_text(
        getattr(span, "span_type", None)
    ).upper()


def span_status_text(span: Any) -> str:
    status = getattr(span, "status", None)

    if status is None:
        return ""

    # MLflow SpanStatus objects commonly expose status_code.
    status_code = getattr(status, "status_code", None)
    if status_code is not None:
        return enum_text(status_code).upper()

    return str(status).upper()


def span_duration_ms(span: Any) -> float:
    start_ns = getattr(span, "start_time_ns", None)
    end_ns = getattr(span, "end_time_ns", None)

    if start_ns is None or end_ns is None:
        return 0.0

    return max(0.0, (end_ns - start_ns) / 1_000_000)


def flatten_dict(
    value: Any,
    prefix: str = "",
) -> dict[str, Any]:
    flattened = {}

    if isinstance(value, dict):
        for key, child in value.items():
            child_key = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )
            flattened.update(
                flatten_dict(child, child_key)
            )
        return flattened

    if isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            child_key = (
                f"{prefix}.{index}"
                if prefix
                else str(index)
            )
            flattened.update(
                flatten_dict(child, child_key)
            )
        return flattened

    flattened[prefix] = value
    return flattened


def possible_json(value: Any) -> Any:
    if not isinstance(value, str):
        return value

    stripped = value.strip()
    if not (
        stripped.startswith("{")
        or stripped.startswith("[")
    ):
        return value

    try:
        return json.loads(stripped)
    except Exception:
        return value


def span_attributes(span: Any) -> dict[str, Any]:
    attributes = getattr(span, "attributes", None)
    parsed = possible_json(attributes)
    return parsed if isinstance(parsed, dict) else {}


def get_span_model_name(span: Any) -> Optional[str]:
    direct = getattr(span, "model_name", None)
    if direct:
        return str(direct)

    flattened = flatten_dict(span_attributes(span))

    preferred_suffixes = (
        "model.name",
        "model_name",
        "request.model",
        "response.model",
        "gen_ai.request.model",
        "gen_ai.response.model",
    )

    for key, value in flattened.items():
        lowered = key.lower()
        if (
            any(lowered.endswith(item) for item in preferred_suffixes)
            and value not in (None, "")
        ):
            return str(value)

    return None


def trace_duration_ms(trace: Any) -> float:
    duration = getattr(
        getattr(trace, "info", None),
        "execution_duration",
        None,
    )
    return float(duration or 0.0)


def all_trace_spans(trace: Any) -> list[Any]:
    data = getattr(trace, "data", None)
    spans = getattr(data, "spans", None)
    return list(spans or [])


trace_summary_rows = []
span_rows = []

for _, eval_row in trace_eval_df.iterrows():
    trace = eval_row["trace"]
    trace_id = eval_row["trace_id"]
    spans = all_trace_spans(trace)

    trace_summary_rows.append(
        {
            "trace_id": trace_id,
            "state": enum_text(
                getattr(trace.info, "state", None)
            ),
            "execution_duration_ms": trace_duration_ms(trace),
            "span_count": len(spans),
            "assessment_count": len(
                getattr(
                    trace.info,
                    "assessments",
                    None,
                )
                or []
            ),
            "token_usage": compact_json(
                getattr(
                    trace.info,
                    "token_usage",
                    None,
                )
            ),
            "tags": compact_json(
                getattr(trace.info, "tags", None)
            ),
            "trace_metadata": compact_json(
                getattr(
                    trace.info,
                    "trace_metadata",
                    None,
                )
            ),
        }
    )

    for span in spans:
        span_rows.append(
            {
                "trace_id": trace_id,
                "span_id": getattr(span, "span_id", None),
                "parent_id": getattr(span, "parent_id", None),
                "name": getattr(span, "name", None),
                "span_type": span_type_text(span),
                "status": span_status_text(span),
                "duration_ms": span_duration_ms(span),
                "model_name": get_span_model_name(span),
                "inputs": compact_json(
                    getattr(span, "inputs", None)
                ),
                "outputs": compact_json(
                    getattr(span, "outputs", None)
                ),
                "attributes": compact_json(
                    span_attributes(span),
                    limit=1200,
                ),
            }
        )

trace_summary_df = pd.DataFrame(trace_summary_rows)
span_inventory_df = pd.DataFrame(span_rows)

print("Trace summary:")
display(trace_summary_df)

if span_inventory_df.empty:
    print("No spans were present in the selected traces.")
else:
    print("\nSpan type/name inventory:")
    span_group_df = (
        span_inventory_df.groupby(
            ["span_type", "name"],
            dropna=False,
        )
        .agg(
            count=("span_id", "count"),
            mean_duration_ms=("duration_ms", "mean"),
            max_duration_ms=("duration_ms", "max"),
            statuses=("status", lambda values: sorted(set(values))),
            models=(
                "model_name",
                lambda values: sorted(
                    {
                        value
                        for value in values
                        if value not in (None, "")
                    }
                ),
            ),
        )
        .reset_index()
        .sort_values(
            ["span_type", "count"],
            ascending=[True, False],
        )
    )
    display(span_group_df)

    print("\nRaw span preview:")
    display(span_inventory_df.head(50))

## 8. Translate spans into reusable evaluation categories

These helpers convert raw trace spans into consistent operational categories.

The logic:

- treats `LLM` spans as model calls
- distinguishes all tool spans from logical leaf tool calls
- separates searches, delegated tasks, file operations, planning, and source validation
- recognizes agent-like work represented as `CHAIN` spans
- extracts token usage when available
- calculates non-overlapping wall-clock time so parallel work is not double-counted

No evaluation runs here. This section defines the interpretation used by the scorers.


In [ ]:
# ---------------------------------------------------------------------
# Span types observed in the AI-Q traces
# ---------------------------------------------------------------------
MODEL_SPAN_TYPES = {"LLM"}
TOOL_SPAN_TYPES = {"TOOL"}
RETRIEVER_SPAN_TYPES = {"RETRIEVER"}
AGENT_SPAN_TYPES = {"AGENT"}
CHAIN_SPAN_TYPES = {"CHAIN"}


# ---------------------------------------------------------------------
# AI-Q tool categories observed in the trace inventory
# ---------------------------------------------------------------------
EXTERNAL_SEARCH_TOOL_NAMES = {
    "tavily_search",
    "paper_search_tool",
}

SEARCH_WRAPPER_TOOL_NAMES = {
    "advanced_web_search_tool",
}

SUBAGENT_TOOL_NAMES = {
    "task",
}

FILE_TOOL_NAMES = {
    "read_file",
    "write_file",
    "edit_file",
    "ls",
}

PLANNING_TOOL_NAMES = {
    "write_todos",
}

SOURCE_VALIDATION_TOOL_NAMES = {
    "get_verified_sources",
}


# AI-Q currently represents these operations as CHAIN spans rather
# than AGENT spans.
AIQ_AGENT_CHAIN_PATTERN = re.compile(
    r"(?:"
    r"agent"
    r"|classifier"
    r")",
    re.IGNORECASE,
)


# ---------------------------------------------------------------------
# Basic span selection
# ---------------------------------------------------------------------
def select_spans_by_type(
    trace: Any,
    span_types: set[str],
) -> list[Any]:
    """Return spans whose normalized span type matches span_types."""
    return [
        span
        for span in all_trace_spans(trace)
        if span_type_text(span) in span_types
    ]


def normalized_span_name(span: Any) -> str:
    """Return a stable string form of the span name."""
    return str(getattr(span, "name", "") or "").strip()


def spans_named(
    spans: Iterable[Any],
    names: set[str],
) -> list[Any]:
    """Select spans by exact span name."""
    return [
        span
        for span in spans
        if normalized_span_name(span) in names
    ]


# ---------------------------------------------------------------------
# Model spans
# ---------------------------------------------------------------------
def model_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return AI-Q model calls.

    The inspected AI-Q traces emit model calls with span_type=LLM.
    """
    spans = select_spans_by_type(
        trace,
        MODEL_SPAN_TYPES,
    )

    return spans, "standard LLM span_type"


def model_spans_by_name(
    trace: Any,
) -> dict[str, list[Any]]:
    """Group model spans by the recorded model name."""
    grouped: dict[str, list[Any]] = {}

    spans, _ = model_spans(trace)

    for span in spans:
        model_name = (
            get_span_model_name(span)
            or normalized_span_name(span)
            or "unknown"
        )

        grouped.setdefault(model_name, []).append(span)

    return grouped


# ---------------------------------------------------------------------
# Tool spans
# ---------------------------------------------------------------------
def all_tool_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return every span emitted with span_type=TOOL.

    This includes wrappers, delegated tasks, file operations, and
    underlying external calls.
    """
    spans = select_spans_by_type(
        trace,
        TOOL_SPAN_TYPES,
    )

    return spans, "all standard TOOL spans"


def leaf_spans(
    trace: Any,
    candidate_spans: Iterable[Any],
) -> list[Any]:
    """Return candidate spans that do not parent any other trace span.

    This avoids counting both a wrapper and the operation nested beneath
    it. For example:

        advanced_web_search_tool
            -> tavily_search

    Only tavily_search is considered a leaf operation.
    """
    all_spans = all_trace_spans(trace)

    parent_span_ids = {
        str(getattr(span, "parent_id", ""))
        for span in all_spans
        if getattr(span, "parent_id", None)
    }

    return [
        span
        for span in candidate_spans
        if str(getattr(span, "span_id", ""))
        not in parent_span_ids
    ]


def leaf_tool_spans(
    trace: Any,
) -> list[Any]:
    """Return non-wrapper TOOL spans."""
    spans, _ = all_tool_spans(trace)
    leaves = leaf_spans(trace, spans)

    # Defensive fallback for an unusual trace structure.
    return leaves if leaves else spans


def tool_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return logical leaf tool calls for operational metrics.

    The operational scorer already calls this helper. Returning leaf
    spans here prevents nested wrapper durations from being counted as
    separate tool-processing time.
    """
    spans = leaf_tool_spans(trace)

    return (
        spans,
        "leaf TOOL spans; nested wrappers excluded",
    )


def tool_category(span: Any) -> str:
    """Classify one AI-Q TOOL span by its exact name."""
    name = normalized_span_name(span)

    if name in EXTERNAL_SEARCH_TOOL_NAMES:
        return "external_search"

    if name in SEARCH_WRAPPER_TOOL_NAMES:
        return "search_wrapper"

    if name in SUBAGENT_TOOL_NAMES:
        return "subagent_task"

    if name in FILE_TOOL_NAMES:
        return "file_operation"

    if name in PLANNING_TOOL_NAMES:
        return "planning"

    if name in SOURCE_VALIDATION_TOOL_NAMES:
        return "source_validation"

    return "other"


def tool_spans_by_category(
    trace: Any,
    *,
    leaf_only: bool = False,
) -> dict[str, list[Any]]:
    """Group AI-Q tool spans into meaningful operational categories."""
    if leaf_only:
        spans = leaf_tool_spans(trace)
    else:
        spans, _ = all_tool_spans(trace)

    grouped: dict[str, list[Any]] = {
        "external_search": [],
        "search_wrapper": [],
        "subagent_task": [],
        "file_operation": [],
        "planning": [],
        "source_validation": [],
        "other": [],
    }

    for span in spans:
        grouped[tool_category(span)].append(span)

    return grouped


def external_search_spans(
    trace: Any,
) -> list[Any]:
    """Return actual web and paper search operations."""
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        EXTERNAL_SEARCH_TOOL_NAMES,
    )


def web_search_spans(
    trace: Any,
) -> list[Any]:
    """Return Tavily search operations."""
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        {"tavily_search"},
    )


def paper_search_spans(
    trace: Any,
) -> list[Any]:
    """Return paper-search operations."""
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        {"paper_search_tool"},
    )


def subagent_task_spans(
    trace: Any,
) -> list[Any]:
    """Return delegated subagent task spans.

    These are intentionally separate from ordinary tool-call latency.
    """
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        SUBAGENT_TOOL_NAMES,
    )


def file_operation_spans(
    trace: Any,
) -> list[Any]:
    """Return read/write/edit/list file operations."""
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        FILE_TOOL_NAMES,
    )


def source_validation_spans(
    trace: Any,
) -> list[Any]:
    """Return source-verification operations."""
    all_tools, _ = all_tool_spans(trace)

    return spans_named(
        all_tools,
        SOURCE_VALIDATION_TOOL_NAMES,
    )


# ---------------------------------------------------------------------
# Retriever, chain, and agent spans
# ---------------------------------------------------------------------
def retriever_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return standard MLflow RETRIEVER spans."""
    spans = select_spans_by_type(
        trace,
        RETRIEVER_SPAN_TYPES,
    )

    return spans, "standard RETRIEVER span_type"


def chain_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return standard CHAIN spans."""
    spans = select_spans_by_type(
        trace,
        CHAIN_SPAN_TYPES,
    )

    return spans, "standard CHAIN span_type"


def agent_spans(
    trace: Any,
) -> tuple[list[Any], str]:
    """Return agent-like workflow spans.

    Prefer standard AGENT spans when present. AI-Q currently records
    agents such as clarifier_agent and intent_classifier as CHAIN spans,
    so those are used when no AGENT spans exist.
    """
    standard_agents = select_spans_by_type(
        trace,
        AGENT_SPAN_TYPES,
    )

    if standard_agents:
        return standard_agents, "standard AGENT span_type"

    chains, _ = chain_spans(trace)

    aiq_agent_chains = [
        span
        for span in chains
        if AIQ_AGENT_CHAIN_PATTERN.search(
            normalized_span_name(span)
        )
    ]

    return (
        aiq_agent_chains,
        "AI-Q agent-like CHAIN spans",
    )


def workflow_spans(
    trace: Any,
) -> list[Any]:
    """Return root or workflow-level spans."""
    return [
        span
        for span in all_trace_spans(trace)
        if (
            normalized_span_name(span) == "<workflow>"
            or normalized_span_name(span).startswith(
                "async_job:"
            )
        )
    ]


# ---------------------------------------------------------------------
# Token extraction
# ---------------------------------------------------------------------
def numeric_value(
    value: Any,
) -> Optional[float]:
    """Convert a scalar token value to float when possible."""
    if isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        stripped = value.strip()

        try:
            return float(stripped)
        except ValueError:
            return None

    return None


def extract_span_token_usage(
    span: Any,
) -> dict[str, float]:
    """Extract token usage from one model span.

    AI and OpenTelemetry integrations can place token usage in span
    attributes, inputs, outputs, or direct usage fields. This checks all
    of those locations.

    For each category, the maximum matching value is used to avoid
    double-counting aliases such as both prompt_tokens and input_tokens.
    """
    flattened_values: dict[str, Any] = {}

    attribute_value = span_attributes(span)
    if attribute_value:
        flattened_values.update(
            flatten_dict(
                attribute_value,
                prefix="attributes",
            )
        )

    for prefix, value in (
        (
            "inputs",
            possible_json(
                getattr(span, "inputs", None)
            ),
        ),
        (
            "outputs",
            possible_json(
                getattr(span, "outputs", None)
            ),
        ),
        (
            "token_usage",
            possible_json(
                getattr(span, "token_usage", None)
            ),
        ),
        (
            "usage",
            possible_json(
                getattr(span, "usage", None)
            ),
        ),
        (
            "llm_token_usage",
            possible_json(
                getattr(span, "llm_token_usage", None)
            ),
        ),
    ):
        if isinstance(value, (dict, list)):
            flattened_values.update(
                flatten_dict(
                    value,
                    prefix=prefix,
                )
            )

    candidates = {
        "input_tokens": [],
        "output_tokens": [],
        "total_tokens": [],
    }

    for key, value in flattened_values.items():
        number = numeric_value(value)

        if number is None:
            continue

        normalized_key = (
            key.lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        if (
            normalized_key.endswith("input_tokens")
            or normalized_key.endswith("prompt_tokens")
        ):
            candidates["input_tokens"].append(number)

        if (
            normalized_key.endswith("output_tokens")
            or normalized_key.endswith("completion_tokens")
        ):
            candidates["output_tokens"].append(number)

        if normalized_key.endswith("total_tokens"):
            candidates["total_tokens"].append(number)

    usage = {
        category: max(values) if values else 0.0
        for category, values in candidates.items()
    }

    if (
        usage["total_tokens"] == 0
        and (
            usage["input_tokens"] > 0
            or usage["output_tokens"] > 0
        )
    ):
        usage["total_tokens"] = (
            usage["input_tokens"]
            + usage["output_tokens"]
        )

    return usage


def aggregate_token_usage(
    spans: Iterable[Any],
) -> dict[str, float]:
    """Aggregate token usage across model spans."""
    aggregate = {
        "input_tokens": 0.0,
        "output_tokens": 0.0,
        "total_tokens": 0.0,
    }

    for span in spans:
        usage = extract_span_token_usage(span)

        for key in aggregate:
            aggregate[key] += usage[key]

    return aggregate


def token_usage_by_model(
    trace: Any,
) -> dict[str, dict[str, float]]:
    """Aggregate token usage separately for each recorded model."""
    return {
        model_name: aggregate_token_usage(spans)
        for model_name, spans in model_spans_by_name(
            trace
        ).items()
    }


# ---------------------------------------------------------------------
# Status and error helpers
# ---------------------------------------------------------------------
def is_error_span(
    span: Any,
) -> bool:
    """Return True only for explicit error-like span statuses."""
    status = span_status_text(span)

    if not status:
        return False

    error_markers = (
        "ERROR",
        "FAIL",
        "CANCEL",
        "TIMEOUT",
        "DEADLINE",
    )

    return any(
        marker in status
        for marker in error_markers
    )


# ---------------------------------------------------------------------
# Timing helpers
# ---------------------------------------------------------------------
def interval_union_duration_ms(
    spans: Iterable[Any],
) -> float:
    """Calculate wall-clock span coverage without double-counting overlap."""
    intervals: list[tuple[int, int]] = []

    for span in spans:
        start_ns = getattr(
            span,
            "start_time_ns",
            None,
        )
        end_ns = getattr(
            span,
            "end_time_ns",
            None,
        )

        if (
            start_ns is None
            or end_ns is None
            or end_ns <= start_ns
        ):
            continue

        intervals.append(
            (
                int(start_ns),
                int(end_ns),
            )
        )

    if not intervals:
        return 0.0

    intervals.sort()

    merged = [
        [
            intervals[0][0],
            intervals[0][1],
        ]
    ]

    for start_ns, end_ns in intervals[1:]:
        previous = merged[-1]

        if start_ns <= previous[1]:
            previous[1] = max(
                previous[1],
                end_ns,
            )
        else:
            merged.append(
                [
                    start_ns,
                    end_ns,
                ]
            )

    total_ns = sum(
        end_ns - start_ns
        for start_ns, end_ns in merged
    )

    return total_ns / 1_000_000


# ---------------------------------------------------------------------
# Duplicate tool-call detection
# ---------------------------------------------------------------------
def normalized_tool_signature(
    span: Any,
) -> str:
    """Create a stable exact signature from a tool name and inputs."""
    payload = {
        "name": normalized_span_name(span),
        "inputs": possible_json(
            getattr(span, "inputs", None)
        ),
    }

    return compact_json(
        payload,
        limit=10000,
    )


# ---------------------------------------------------------------------
# Trace-depth calculation
# ---------------------------------------------------------------------
def maximum_trace_depth(
    trace: Any,
) -> int:
    """Calculate maximum parent-child depth in the trace."""
    spans = all_trace_spans(trace)

    parent_by_span_id = {
        str(
            getattr(
                span,
                "span_id",
                "",
            )
        ): (
            str(
                getattr(
                    span,
                    "parent_id",
                    "",
                )
            )
            if getattr(
                span,
                "parent_id",
                None,
            )
            else None
        )
        for span in spans
    }

    maximum_depth = 0

    for span in spans:
        current_span_id = str(
            getattr(
                span,
                "span_id",
                "",
            )
        )

        depth = 1
        visited: set[str] = set()

        while (
            current_span_id
            and current_span_id not in visited
        ):
            visited.add(current_span_id)

            parent_id = parent_by_span_id.get(
                current_span_id
            )

            if not parent_id:
                break

            depth += 1
            current_span_id = parent_id

        maximum_depth = max(
            maximum_depth,
            depth,
        )

    return maximum_depth


print("AI-Q trace classification helpers loaded.")
print("  Model spans: LLM")
print("  Generic tool metrics: leaf TOOL spans")
print(
    "  Tool categories:",
    [
        "external_search",
        "search_wrapper",
        "subagent_task",
        "file_operation",
        "planning",
        "source_validation",
        "other",
    ],
)

## 9. Define deterministic operational scorers

The operational scorer evaluates the execution that already happened. It does not call AI-Q again and does not make external model calls.

For each trace, it records assessments covering:

- end-to-end duration, span count, and trace depth
- overall and per-model call counts and latency
- cumulative time versus non-overlapping wall-clock time
- model, tool, and delegated-task parallelism
- web search, paper search, file, planning, and source-validation activity
- duplicate tool-call patterns
- explicit error statuses
- token usage when available

These are measurements, not an overall quality verdict. A high search count or long duration is not automatically good or bad until compared with a scenario-specific target.

> **Important:** Cumulative span time can exceed end-to-end duration because nested and concurrent spans overlap.


In [ ]:
from mlflow.entities import Feedback, Trace
from mlflow.genai import evaluate, scorer


@scorer
def aiq_trace_operational_metrics(
    trace: Trace,
) -> list[Feedback]:
    # ---------------------------------------------------------------
    # Core span classification
    # ---------------------------------------------------------------
    models, model_detection = model_spans(trace)
    tools, tool_detection = tool_spans(trace)
    all_tools, _ = all_tool_spans(trace)

    retrievers, retriever_detection = retriever_spans(trace)
    agents, agent_detection = agent_spans(trace)
    all_spans = all_trace_spans(trace)

    # ---------------------------------------------------------------
    # AI-Q-specific tool categories
    # ---------------------------------------------------------------
    tool_categories = tool_spans_by_category(
        trace,
        leaf_only=False,
    )

    web_searches = web_search_spans(trace)
    paper_searches = paper_search_spans(trace)

    external_searches = (
        web_searches
        + paper_searches
    )

    subagent_tasks = subagent_task_spans(trace)
    file_operations = file_operation_spans(trace)
    source_validations = source_validation_spans(trace)

    planning_tools = tool_categories["planning"]
    search_wrappers = tool_categories["search_wrapper"]
    other_tools = tool_categories["other"]

    # ---------------------------------------------------------------
    # Model-specific groupings
    # ---------------------------------------------------------------
    model_groups = model_spans_by_name(trace)

    nemotron_spans = [
        span
        for model_name, grouped_spans
        in model_groups.items()
        if "nemotron" in model_name.lower()
        for span in grouped_spans
    ]

    gpt_oss_spans = [
        span
        for model_name, grouped_spans
        in model_groups.items()
        if "gpt-oss" in model_name.lower()
        for span in grouped_spans
    ]

    # ---------------------------------------------------------------
    # End-to-end trace timing
    # ---------------------------------------------------------------
    total_ms = trace_duration_ms(trace)

    # ---------------------------------------------------------------
    # Model timing
    # ---------------------------------------------------------------
    model_durations = [
        span_duration_ms(span)
        for span in models
    ]

    cumulative_model_ms = sum(model_durations)

    model_wall_clock_ms = interval_union_duration_ms(
        models
    )

    model_parallelism = (
        cumulative_model_ms / model_wall_clock_ms
        if model_wall_clock_ms > 0
        else 0.0
    )

    # ---------------------------------------------------------------
    # Leaf tool timing
    #
    # tool_spans() returns leaf TOOL spans, excluding wrappers that
    # contain another operation.
    # ---------------------------------------------------------------
    tool_durations = [
        span_duration_ms(span)
        for span in tools
    ]

    cumulative_tool_ms = sum(tool_durations)

    tool_wall_clock_ms = interval_union_duration_ms(
        tools
    )

    tool_parallelism = (
        cumulative_tool_ms / tool_wall_clock_ms
        if tool_wall_clock_ms > 0
        else 0.0
    )

    # ---------------------------------------------------------------
    # External search timing
    # ---------------------------------------------------------------
    external_search_durations = [
        span_duration_ms(span)
        for span in external_searches
    ]

    external_search_wall_clock_ms = (
        interval_union_duration_ms(
            external_searches
        )
    )

    # ---------------------------------------------------------------
    # Delegated subagent timing
    # ---------------------------------------------------------------
    subagent_task_durations = [
        span_duration_ms(span)
        for span in subagent_tasks
    ]

    subagent_task_wall_clock_ms = (
        interval_union_duration_ms(
            subagent_tasks
        )
    )

    subagent_parallelism = (
        sum(subagent_task_durations)
        / subagent_task_wall_clock_ms
        if subagent_task_wall_clock_ms > 0
        else 0.0
    )

    # ---------------------------------------------------------------
    # Model-specific timing
    # ---------------------------------------------------------------
    nemotron_durations = [
        span_duration_ms(span)
        for span in nemotron_spans
    ]

    gpt_oss_durations = [
        span_duration_ms(span)
        for span in gpt_oss_spans
    ]

    # ---------------------------------------------------------------
    # Duplicate leaf tool-call analysis
    # ---------------------------------------------------------------
    tool_signatures = [
        normalized_tool_signature(span)
        for span in tools
    ]

    unique_tool_count = len(
        set(tool_signatures)
    )

    duplicate_tool_count = (
        len(tool_signatures)
        - unique_tool_count
    )

    unique_tool_ratio = (
        unique_tool_count
        / len(tool_signatures)
        if tool_signatures
        else 1.0
    )

    # ---------------------------------------------------------------
    # Error analysis
    # ---------------------------------------------------------------
    error_count = sum(
        is_error_span(span)
        for span in all_spans
    )

    model_error_count = sum(
        is_error_span(span)
        for span in models
    )

    tool_error_count = sum(
        is_error_span(span)
        for span in tools
    )

    external_search_error_count = sum(
        is_error_span(span)
        for span in external_searches
    )

    subagent_task_error_count = sum(
        is_error_span(span)
        for span in subagent_tasks
    )

    # ---------------------------------------------------------------
    # Token usage
    # ---------------------------------------------------------------
    usage = aggregate_token_usage(
        models
    )

    nemotron_usage = aggregate_token_usage(
        nemotron_spans
    )

    gpt_oss_usage = aggregate_token_usage(
        gpt_oss_spans
    )

    # ---------------------------------------------------------------
    # Operational feedback
    # ---------------------------------------------------------------
    return [
        Feedback(
            name="end_to_end_seconds",
            value=round(
                total_ms / 1000,
                4,
            ),
            rationale=(
                "Observed wall-clock duration of the complete trace."
            ),
        ),
        Feedback(
            name="span_count",
            value=len(all_spans),
            rationale=(
                "Total number of spans in the trace."
            ),
        ),
        Feedback(
            name="trace_max_depth",
            value=maximum_trace_depth(trace),
            rationale=(
                "Maximum parent-child span depth in the trace."
            ),
        ),

        # -----------------------------------------------------------
        # Overall model metrics
        # -----------------------------------------------------------
        Feedback(
            name="model_call_count",
            value=len(models),
            rationale=(
                "Model spans identified using "
                f"{model_detection}."
            ),
        ),
        Feedback(
            name="cumulative_model_seconds",
            value=round(
                cumulative_model_ms / 1000,
                4,
            ),
            rationale=(
                "Sum of model span durations. Concurrent calls are "
                "counted individually."
            ),
        ),
        Feedback(
            name="model_wall_clock_seconds",
            value=round(
                model_wall_clock_ms / 1000,
                4,
            ),
            rationale=(
                "Union of model span intervals without "
                "double-counting overlapping calls."
            ),
        ),
        Feedback(
            name="model_parallelism_factor",
            value=round(
                model_parallelism,
                4,
            ),
            rationale=(
                "Cumulative model time divided by non-overlapping "
                "model wall-clock time. Values above 1 indicate "
                "concurrent or overlapping model calls."
            ),
        ),
        Feedback(
            name="mean_model_call_seconds",
            value=round(
                (
                    cumulative_model_ms
                    / len(model_durations)
                    / 1000
                )
                if model_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average duration across identified model spans."
            ),
        ),
        Feedback(
            name="slowest_model_call_seconds",
            value=round(
                max(
                    model_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest identified model span."
            ),
        ),
        Feedback(
            name="model_error_count",
            value=model_error_count,
            rationale=(
                "Identified model spans with an explicit "
                "error-like status."
            ),
        ),

        # -----------------------------------------------------------
        # Nemotron metrics
        # -----------------------------------------------------------
        Feedback(
            name="nemotron_call_count",
            value=len(nemotron_spans),
            rationale=(
                "Number of LLM spans whose recorded model name "
                "contains 'nemotron'."
            ),
        ),
        Feedback(
            name="nemotron_mean_call_seconds",
            value=round(
                (
                    sum(nemotron_durations)
                    / len(nemotron_durations)
                    / 1000
                )
                if nemotron_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average observed duration of Nemotron model calls."
            ),
        ),
        Feedback(
            name="nemotron_cumulative_seconds",
            value=round(
                sum(nemotron_durations) / 1000,
                4,
            ),
            rationale=(
                "Sum of all Nemotron model-span durations."
            ),
        ),
        Feedback(
            name="nemotron_slowest_call_seconds",
            value=round(
                max(
                    nemotron_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest Nemotron model call."
            ),
        ),
        Feedback(
            name="nemotron_input_tokens",
            value=int(
                nemotron_usage["input_tokens"]
            ),
            rationale=(
                "Input or prompt tokens extracted from Nemotron "
                "model spans when available."
            ),
        ),
        Feedback(
            name="nemotron_output_tokens",
            value=int(
                nemotron_usage["output_tokens"]
            ),
            rationale=(
                "Output or completion tokens extracted from "
                "Nemotron model spans when available."
            ),
        ),
        Feedback(
            name="nemotron_total_tokens",
            value=int(
                nemotron_usage["total_tokens"]
            ),
            rationale=(
                "Total tokens extracted from Nemotron model spans "
                "when available."
            ),
        ),

        # -----------------------------------------------------------
        # GPT-OSS metrics
        # -----------------------------------------------------------
        Feedback(
            name="gpt_oss_call_count",
            value=len(gpt_oss_spans),
            rationale=(
                "Number of LLM spans whose recorded model name "
                "contains 'gpt-oss'."
            ),
        ),
        Feedback(
            name="gpt_oss_mean_call_seconds",
            value=round(
                (
                    sum(gpt_oss_durations)
                    / len(gpt_oss_durations)
                    / 1000
                )
                if gpt_oss_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average observed duration of GPT-OSS model calls."
            ),
        ),
        Feedback(
            name="gpt_oss_cumulative_seconds",
            value=round(
                sum(gpt_oss_durations) / 1000,
                4,
            ),
            rationale=(
                "Sum of all GPT-OSS model-span durations."
            ),
        ),
        Feedback(
            name="gpt_oss_slowest_call_seconds",
            value=round(
                max(
                    gpt_oss_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest GPT-OSS model call."
            ),
        ),
        Feedback(
            name="gpt_oss_input_tokens",
            value=int(
                gpt_oss_usage["input_tokens"]
            ),
            rationale=(
                "Input or prompt tokens extracted from GPT-OSS "
                "model spans when available."
            ),
        ),
        Feedback(
            name="gpt_oss_output_tokens",
            value=int(
                gpt_oss_usage["output_tokens"]
            ),
            rationale=(
                "Output or completion tokens extracted from "
                "GPT-OSS model spans when available."
            ),
        ),
        Feedback(
            name="gpt_oss_total_tokens",
            value=int(
                gpt_oss_usage["total_tokens"]
            ),
            rationale=(
                "Total tokens extracted from GPT-OSS model spans "
                "when available."
            ),
        ),

        # -----------------------------------------------------------
        # Leaf tool metrics
        # -----------------------------------------------------------
        Feedback(
            name="tool_call_count",
            value=len(tools),
            rationale=(
                "Leaf tool spans identified using "
                f"{tool_detection}. Nested wrapper spans are excluded."
            ),
        ),
        Feedback(
            name="all_tool_span_count",
            value=len(all_tools),
            rationale=(
                "All spans recorded with TOOL type, including "
                "wrappers, delegated tasks, file operations, and "
                "underlying external calls."
            ),
        ),
        Feedback(
            name="cumulative_tool_seconds",
            value=round(
                cumulative_tool_ms / 1000,
                4,
            ),
            rationale=(
                "Sum of leaf tool-span durations. Concurrent calls "
                "are counted individually."
            ),
        ),
        Feedback(
            name="tool_wall_clock_seconds",
            value=round(
                tool_wall_clock_ms / 1000,
                4,
            ),
            rationale=(
                "Union of leaf tool-span intervals without "
                "double-counting overlapping calls."
            ),
        ),
        Feedback(
            name="tool_parallelism_factor",
            value=round(
                tool_parallelism,
                4,
            ),
            rationale=(
                "Cumulative leaf tool time divided by non-overlapping "
                "tool wall-clock time. Values above 1 indicate "
                "overlapping tool execution."
            ),
        ),
        Feedback(
            name="mean_tool_call_seconds",
            value=round(
                (
                    cumulative_tool_ms
                    / len(tool_durations)
                    / 1000
                )
                if tool_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average duration across identified leaf tool spans."
            ),
        ),
        Feedback(
            name="slowest_tool_call_seconds",
            value=round(
                max(
                    tool_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest identified leaf tool span."
            ),
        ),
        Feedback(
            name="tool_error_count",
            value=tool_error_count,
            rationale=(
                "Leaf tool spans with an explicit error-like status."
            ),
        ),
        Feedback(
            name="unique_tool_call_ratio",
            value=round(
                unique_tool_ratio,
                4,
            ),
            rationale=(
                f"{unique_tool_count} unique exact tool name/input "
                f"combinations across {len(tools)} leaf tool calls."
            ),
        ),
        Feedback(
            name="duplicate_tool_call_count",
            value=duplicate_tool_count,
            rationale=(
                "Number of leaf tool calls whose exact tool-name and "
                "input signature duplicated another call."
            ),
        ),

        # -----------------------------------------------------------
        # Search metrics
        # -----------------------------------------------------------
        Feedback(
            name="search_wrapper_count",
            value=len(search_wrappers),
            rationale=(
                "Number of advanced_web_search_tool wrapper spans."
            ),
        ),
        Feedback(
            name="web_search_count",
            value=len(web_searches),
            rationale=(
                "Number of underlying tavily_search tool calls."
            ),
        ),
        Feedback(
            name="paper_search_count",
            value=len(paper_searches),
            rationale=(
                "Number of paper_search_tool calls."
            ),
        ),
        Feedback(
            name="external_search_count",
            value=len(external_searches),
            rationale=(
                "Combined Tavily web searches and paper searches."
            ),
        ),
        Feedback(
            name="external_search_cumulative_seconds",
            value=round(
                sum(external_search_durations) / 1000,
                4,
            ),
            rationale=(
                "Sum of Tavily and paper-search span durations. "
                "Concurrent searches are counted individually."
            ),
        ),
        Feedback(
            name="external_search_wall_clock_seconds",
            value=round(
                external_search_wall_clock_ms / 1000,
                4,
            ),
            rationale=(
                "Wall-clock coverage of external searches without "
                "double-counting overlapping calls."
            ),
        ),
        Feedback(
            name="mean_external_search_seconds",
            value=round(
                (
                    sum(external_search_durations)
                    / len(external_search_durations)
                    / 1000
                )
                if external_search_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average duration across Tavily and paper searches."
            ),
        ),
        Feedback(
            name="slowest_external_search_seconds",
            value=round(
                max(
                    external_search_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest web or paper search."
            ),
        ),
        Feedback(
            name="external_search_error_count",
            value=external_search_error_count,
            rationale=(
                "External-search spans with an explicit "
                "error-like status."
            ),
        ),

        # -----------------------------------------------------------
        # Delegated subagent metrics
        # -----------------------------------------------------------
        Feedback(
            name="subagent_task_count",
            value=len(subagent_tasks),
            rationale=(
                "Number of delegated research task spans."
            ),
        ),
        Feedback(
            name="subagent_task_cumulative_seconds",
            value=round(
                sum(subagent_task_durations) / 1000,
                4,
            ),
            rationale=(
                "Sum of delegated subagent task durations. "
                "Concurrent tasks are counted individually."
            ),
        ),
        Feedback(
            name="subagent_task_wall_clock_seconds",
            value=round(
                subagent_task_wall_clock_ms / 1000,
                4,
            ),
            rationale=(
                "Wall-clock coverage of delegated tasks without "
                "double-counting overlapping execution."
            ),
        ),
        Feedback(
            name="subagent_parallelism_factor",
            value=round(
                subagent_parallelism,
                4,
            ),
            rationale=(
                "Cumulative delegated-task duration divided by "
                "non-overlapping task wall-clock time."
            ),
        ),
        Feedback(
            name="mean_subagent_task_seconds",
            value=round(
                (
                    sum(subagent_task_durations)
                    / len(subagent_task_durations)
                    / 1000
                )
                if subagent_task_durations
                else 0.0,
                4,
            ),
            rationale=(
                "Average duration across delegated research tasks."
            ),
        ),
        Feedback(
            name="slowest_subagent_task_seconds",
            value=round(
                max(
                    subagent_task_durations,
                    default=0.0,
                )
                / 1000,
                4,
            ),
            rationale=(
                "Duration of the slowest delegated research task."
            ),
        ),
        Feedback(
            name="subagent_task_error_count",
            value=subagent_task_error_count,
            rationale=(
                "Delegated task spans with an explicit "
                "error-like status."
            ),
        ),

        # -----------------------------------------------------------
        # Internal workflow-tool metrics
        # -----------------------------------------------------------
        Feedback(
            name="file_operation_count",
            value=len(file_operations),
            rationale=(
                "Combined read_file, write_file, edit_file, and "
                "ls operations."
            ),
        ),
        Feedback(
            name="planning_tool_call_count",
            value=len(planning_tools),
            rationale=(
                "Number of write_todos planning operations."
            ),
        ),
        Feedback(
            name="source_validation_call_count",
            value=len(source_validations),
            rationale=(
                "Number of get_verified_sources operations."
            ),
        ),
        Feedback(
            name="other_tool_span_count",
            value=len(other_tools),
            rationale=(
                "TOOL spans that did not match one of the defined "
                "AI-Q tool categories."
            ),
        ),

        # -----------------------------------------------------------
        # Retriever and agent metrics
        # -----------------------------------------------------------
        Feedback(
            name="retriever_call_count",
            value=len(retrievers),
            rationale=(
                "Retriever spans identified using "
                f"{retriever_detection}."
            ),
        ),
        Feedback(
            name="agent_span_count",
            value=len(agents),
            rationale=(
                "Agent-like spans identified using "
                f"{agent_detection}."
            ),
        ),

        # -----------------------------------------------------------
        # Overall error metrics
        # -----------------------------------------------------------
        Feedback(
            name="error_span_count",
            value=error_count,
            rationale=(
                "Total spans with an explicit error-like status."
            ),
        ),
        Feedback(
            name="all_spans_successful",
            value=error_count == 0,
            rationale=(
                "True when no span has an explicit error-like status."
            ),
        ),

        # -----------------------------------------------------------
        # Overall token metrics
        # -----------------------------------------------------------
        Feedback(
            name="model_input_tokens",
            value=int(
                usage["input_tokens"]
            ),
            rationale=(
                "Input or prompt tokens extracted from all model "
                "spans when available."
            ),
        ),
        Feedback(
            name="model_output_tokens",
            value=int(
                usage["output_tokens"]
            ),
            rationale=(
                "Output or completion tokens extracted from all "
                "model spans when available."
            ),
        ),
        Feedback(
            name="model_total_tokens",
            value=int(
                usage["total_tokens"]
            ),
            rationale=(
                "Total tokens extracted from all model spans "
                "when available."
            ),
        ),
    ]


evaluation_scorers = [
    aiq_trace_operational_metrics,
]


print(
    "Configured deterministic scorers:",
    [
        item.name
        for item in evaluation_scorers
    ],
)

## 10. Optional next step: turn operational measurements into expectation checks

Our primary evaluation path in this notebook for the AI quickstart stops at **measurement**: it records what happened during each AI-Q execution. After you have meaningful requirements or a baseline from several comparable traces, you can optionally turn selected measurements into deterministic pass/fail checks.

MLflow passes each record's `expectations` dictionary into a custom scorer. The scorer below looks for AI-Q-specific operational expectation fields and emits checks only for fields that are present.

### When to use this

Use expectation checks when the threshold represents a real requirement, such as:

- an agreed latency budget
- a maximum error count
- a known tool that must or must not be used for a scenario
- a model/tool-call budget derived from cost or capacity requirements
- a minimum search requirement that is meaningful for a specific research task

Avoid choosing thresholds from a single trace just because that trace happened to produce a particular value. A useful workflow is:

1. Run the operational evaluation on several representative traces.
2. Review the distributions and identify normal ranges.
3. Combine those baselines with product requirements, SLOs, cost constraints, or domain-expert guidance.
4. Add only the expectations that are meaningful for each dataset record.
5. Set `ENABLE_EXPECTATION_CHECKS=True`.
6. Rerun the evaluation section and review the new boolean assessments beside the operational measurements.

### Supported expectation fields

| Expectation field | Resulting check |
|---|---|
| `max_total_seconds` | End-to-end trace duration stays within the latency budget |
| `min_model_calls` / `max_model_calls` | Model-call count stays within the expected range |
| `min_tool_calls` / `max_tool_calls` | Logical leaf tool-call count stays within the expected range |
| `min_all_tool_spans` / `max_all_tool_spans` | Total TOOL-span count stays within the expected range |
| `required_tool_names` | Required tool/span names were observed |
| `forbidden_tool_names` | Forbidden tool/span names were not observed |
| `min_web_searches` / `max_web_searches` | AI-Q web-search activity stays within the expected range |
| `min_paper_searches` / `max_paper_searches` | Paper-search activity stays within the expected range |
| `max_error_spans` | Explicit error-like spans stay within the allowed budget |

Example expectations for one dataset record:

```python
{
    "max_total_seconds": 1200,
    "max_model_calls": 100,
    "min_web_searches": 5,
    "required_tool_names": [
        "advanced_web_search_tool",
        "get_verified_sources",
    ],
    "max_error_spans": 0,
}
```

Different records can carry different expectations. If a field is absent, this scorer does not invent a target or emit a check for it.

> **Interpretation:** Operational metrics answer **what happened**. Expectation checks answer **whether selected behavior met requirements you explicitly defined**. They still do not establish that the final research answer is factually correct.


In [ ]:
@scorer
def aiq_trace_expectation_checks(
    trace: Trace,
    expectations: Optional[dict[str, Any]] = None,
) -> list[Feedback]:
    expectations = expectations or {}

    models, _ = model_spans(trace)

    # Logical leaf tool calls used for count-based expectations.
    tools, tool_detection = tool_spans(trace)

    # All TOOL spans used for required/forbidden name checks so that
    # wrappers and delegated task spans are not missed.
    all_tools, _ = all_tool_spans(trace)

    all_spans = all_trace_spans(trace)

    total_seconds = trace_duration_ms(trace) / 1000

    leaf_tool_names = [
        normalized_span_name(span)
        for span in tools
    ]

    all_tool_names = [
        normalized_span_name(span)
        for span in all_tools
    ]

    lowered_all_tool_names = [
        name.lower()
        for name in all_tool_names
    ]

    error_count = sum(
        is_error_span(span)
        for span in all_spans
    )

    feedback: list[Feedback] = []

    def add_boolean(
        name: str,
        passed: bool,
        rationale: str,
    ) -> None:
        feedback.append(
            Feedback(
                name=name,
                value=passed,
                rationale=rationale,
            )
        )

    # ---------------------------------------------------------------
    # End-to-end latency
    # ---------------------------------------------------------------
    max_total_seconds = expectations.get(
        "max_total_seconds"
    )

    if max_total_seconds is not None:
        maximum = float(max_total_seconds)

        add_boolean(
            "latency_budget_met",
            total_seconds <= maximum,
            (
                f"Actual: {total_seconds:.2f}s; "
                f"maximum: {maximum:.2f}s."
            ),
        )

    # ---------------------------------------------------------------
    # Model-call budgets
    # ---------------------------------------------------------------
    min_model_calls = expectations.get(
        "min_model_calls"
    )

    if min_model_calls is not None:
        minimum = int(min_model_calls)

        add_boolean(
            "minimum_model_calls_met",
            len(models) >= minimum,
            (
                f"Actual: {len(models)}; "
                f"minimum: {minimum}."
            ),
        )

    max_model_calls = expectations.get(
        "max_model_calls"
    )

    if max_model_calls is not None:
        maximum = int(max_model_calls)

        add_boolean(
            "model_call_budget_met",
            len(models) <= maximum,
            (
                f"Actual: {len(models)}; "
                f"maximum: {maximum}."
            ),
        )

    # ---------------------------------------------------------------
    # Leaf tool-call budgets
    #
    # These count logical leaf TOOL spans. Nested wrapper spans are
    # intentionally excluded.
    # ---------------------------------------------------------------
    min_tool_calls = expectations.get(
        "min_tool_calls"
    )

    if min_tool_calls is not None:
        minimum = int(min_tool_calls)

        add_boolean(
            "minimum_tool_calls_met",
            len(tools) >= minimum,
            (
                f"Actual leaf tool calls: {len(tools)}; "
                f"minimum: {minimum}. "
                f"Detection: {tool_detection}."
            ),
        )

    max_tool_calls = expectations.get(
        "max_tool_calls"
    )

    if max_tool_calls is not None:
        maximum = int(max_tool_calls)

        add_boolean(
            "tool_call_budget_met",
            len(tools) <= maximum,
            (
                f"Actual leaf tool calls: {len(tools)}; "
                f"maximum: {maximum}. "
                f"Detection: {tool_detection}."
            ),
        )

    # ---------------------------------------------------------------
    # Optional total TOOL-span budgets
    #
    # These include wrappers, delegated tasks, file operations, and
    # underlying tool calls.
    # ---------------------------------------------------------------
    min_all_tool_spans = expectations.get(
        "min_all_tool_spans"
    )

    if min_all_tool_spans is not None:
        minimum = int(min_all_tool_spans)

        add_boolean(
            "minimum_all_tool_spans_met",
            len(all_tools) >= minimum,
            (
                f"Actual TOOL spans: {len(all_tools)}; "
                f"minimum: {minimum}."
            ),
        )

    max_all_tool_spans = expectations.get(
        "max_all_tool_spans"
    )

    if max_all_tool_spans is not None:
        maximum = int(max_all_tool_spans)

        add_boolean(
            "all_tool_span_budget_met",
            len(all_tools) <= maximum,
            (
                f"Actual TOOL spans: {len(all_tools)}; "
                f"maximum: {maximum}."
            ),
        )

    # ---------------------------------------------------------------
    # Required and forbidden tools
    #
    # Search all TOOL spans rather than only leaf tools. This allows
    # expectations to match AI-Q wrappers such as
    # advanced_web_search_tool and delegated task spans.
    # ---------------------------------------------------------------
    required_tool_names = expectations.get(
        "required_tool_names",
        [],
    )

    if required_tool_names:
        missing = [
            expected_name
            for expected_name in required_tool_names
            if not any(
                str(expected_name).lower()
                in actual_name
                for actual_name in lowered_all_tool_names
            )
        ]

        add_boolean(
            "required_tools_used",
            not missing,
            (
                f"Observed TOOL span names: "
                f"{sorted(set(all_tool_names))}. "
                f"Missing required matches: {missing}."
            ),
        )

    forbidden_tool_names = expectations.get(
        "forbidden_tool_names",
        [],
    )

    if forbidden_tool_names:
        found = [
            forbidden_name
            for forbidden_name in forbidden_tool_names
            if any(
                str(forbidden_name).lower()
                in actual_name
                for actual_name in lowered_all_tool_names
            )
        ]

        add_boolean(
            "forbidden_tools_avoided",
            not found,
            (
                f"Observed TOOL span names: "
                f"{sorted(set(all_tool_names))}. "
                f"Forbidden matches found: {found}."
            ),
        )

    # ---------------------------------------------------------------
    # AI-Q-specific search expectations
    # ---------------------------------------------------------------
    web_searches = web_search_spans(trace)
    paper_searches = paper_search_spans(trace)

    min_web_searches = expectations.get(
        "min_web_searches"
    )

    if min_web_searches is not None:
        minimum = int(min_web_searches)

        add_boolean(
            "minimum_web_searches_met",
            len(web_searches) >= minimum,
            (
                f"Actual Tavily searches: {len(web_searches)}; "
                f"minimum: {minimum}."
            ),
        )

    max_web_searches = expectations.get(
        "max_web_searches"
    )

    if max_web_searches is not None:
        maximum = int(max_web_searches)

        add_boolean(
            "web_search_budget_met",
            len(web_searches) <= maximum,
            (
                f"Actual Tavily searches: {len(web_searches)}; "
                f"maximum: {maximum}."
            ),
        )

    min_paper_searches = expectations.get(
        "min_paper_searches"
    )

    if min_paper_searches is not None:
        minimum = int(min_paper_searches)

        add_boolean(
            "minimum_paper_searches_met",
            len(paper_searches) >= minimum,
            (
                f"Actual paper searches: {len(paper_searches)}; "
                f"minimum: {minimum}."
            ),
        )

    max_paper_searches = expectations.get(
        "max_paper_searches"
    )

    if max_paper_searches is not None:
        maximum = int(max_paper_searches)

        add_boolean(
            "paper_search_budget_met",
            len(paper_searches) <= maximum,
            (
                f"Actual paper searches: {len(paper_searches)}; "
                f"maximum: {maximum}."
            ),
        )

    # ---------------------------------------------------------------
    # Error budget
    # ---------------------------------------------------------------
    max_error_spans = expectations.get(
        "max_error_spans"
    )

    if max_error_spans is not None:
        maximum = int(max_error_spans)

        add_boolean(
            "error_span_budget_met",
            error_count <= maximum,
            (
                f"Actual: {error_count}; "
                f"maximum: {maximum}."
            ),
        )

    return feedback


# Keep the scorer out of the main quickstart path unless the dataset
# contains meaningful, deliberately chosen expectation fields.
if ENABLE_EXPECTATION_CHECKS:
    evaluation_scorers.append(aiq_trace_expectation_checks)
    print(
        "Expectation checks enabled:",
        aiq_trace_expectation_checks.name,
    )
else:
    print(
        "Expectation checks are defined but disabled. "
        "Set ENABLE_EXPECTATION_CHECKS=True after adding "
        "record-level expectations to the evaluation dataset."
    )

## 11. Run the evaluation and log it to MLflow

This is the main execution step.

MLflow receives the historical trace objects, runs the configured deterministic scorers, and creates a new evaluation run in the source `rh-research` experiment. There is no `predict_fn` because the application is not rerun.

The output confirms:

- the destination experiment and evaluation run ID
- the number of evaluated records
- the configured scorer names
- aggregate metrics across the evaluation set
- per-trace assessment columns
- the evaluated traces associated with the evaluation run

With one trace, metrics ending in `/mean` equal that trace's value. With multiple traces, they are aggregate means across the evaluation set.

If `ENABLE_EXPECTATION_CHECKS=True`, the same evaluation run also includes the optional boolean expectation assessments defined in section 10.

> **Demo cue:** Open the new run in MLflow and show the operational metrics, then open a linked trace to show its per-trace assessments.


In [ ]:
from datetime import datetime, timezone

from mlflow import MlflowClient


operational_evaluation_result = None
operational_run_id = None

client = MlflowClient()


def display_evaluation_result(
    result: Any,
    label: str,
) -> None:
    """Display the result returned by mlflow.genai.evaluate()."""
    print(f"\n{label} evaluation result type:")
    print(type(result))

    metrics = getattr(
        result,
        "metrics",
        None,
    )

    print(
        f"{label} aggregate metric count:",
        len(metrics) if metrics is not None else 0,
    )

    if metrics:
        display(
            pd.DataFrame(
                [
                    {
                        "metric": key,
                        "value": value,
                    }
                    for key, value in sorted(
                        metrics.items()
                    )
                ]
            )
        )
    else:
        print(
            "No aggregate metrics were returned by "
            "the evaluation object."
        )

    result_df = getattr(
        result,
        "result_df",
        None,
    )

    if result_df is not None:
        print(
            f"\n{label} per-trace result shape:",
            result_df.shape,
        )
        display(result_df)
    else:
        print(
            f"{label} result_df is not available."
        )


def verify_logged_run(
    run_id: str,
    expected_experiment_id: str,
    label: str,
) -> None:
    """Verify that the evaluation run exists in the MLflow backend."""
    logged_run = client.get_run(run_id)

    print(f"\n{label} MLflow run verification:")
    print("  Run ID:", logged_run.info.run_id)
    print("  Run name:", logged_run.info.run_name)
    print("  Run status:", logged_run.info.status)
    print(
        "  Experiment ID:",
        logged_run.info.experiment_id,
    )
    print(
        "  Expected experiment ID:",
        expected_experiment_id,
    )
    print(
        "  Logged classic metric count:",
        len(logged_run.data.metrics),
    )
    print(
        "  Run type:",
        logged_run.data.tags.get(
            "mlflow.runType",
            "<not set>",
        ),
    )

    if (
        str(logged_run.info.experiment_id)
        != str(expected_experiment_id)
    ):
        print(
            "\nWARNING: The run was logged to a different "
            "experiment than expected."
        )

    if logged_run.data.metrics:
        display(
            pd.DataFrame(
                [
                    {
                        "metric": key,
                        "value": value,
                    }
                    for key, value in sorted(
                        logged_run.data.metrics.items()
                    )
                ]
            )
        )

    # Evaluation of existing traces should associate evaluation traces
    # with the evaluation run. This confirms whether that happened.
    try:
        evaluation_traces_df = mlflow.search_traces(
            run_id=run_id,
        )

        print(
            "  Evaluation traces associated with run:",
            len(evaluation_traces_df),
        )

        if not evaluation_traces_df.empty:
            visible_columns = [
                column
                for column in [
                    "trace_id",
                    "state",
                    "request_time",
                    "execution_duration",
                    "assessments",
                ]
                if column
                in evaluation_traces_df.columns
            ]

            display(
                evaluation_traces_df[
                    visible_columns
                ]
            )

    except Exception as exc:
        print(
            "  Could not query traces by run ID:",
            type(exc).__name__,
            str(exc),
        )


# Prevent evaluate() from silently reusing an unrelated active run.
existing_run = mlflow.active_run()

if existing_run is not None:
    print(
        "Ending previously active run before evaluation:",
        existing_run.info.run_id,
    )
    mlflow.end_run()

evaluation_experiment = mlflow.set_experiment(
    experiment_id=str(TRACE_EXPERIMENT_ID)
)

if RUN_OPERATIONAL_EVALUATION:
    operational_run_name = (
        "aiq-operational-trace-evaluation-"
        + datetime.now(timezone.utc).strftime(
            "%Y%m%d-%H%M%S"
        )
    )

    print("\nStarting operational evaluation:")
    print(
        "  Experiment name:",
        evaluation_experiment.name,
    )
    print(
        "  Experiment ID:",
        evaluation_experiment.experiment_id,
    )
    print(
        "  Evaluation records:",
        len(evaluation_data_df),
    )
    print(
        "  Evaluation columns:",
        list(evaluation_data_df.columns),
    )
    print(
        "  Scorers:",
        [
            item.name
            for item in evaluation_scorers
        ],
    )

    with mlflow.start_run(
        experiment_id=str(TRACE_EXPERIMENT_ID),
        run_name=operational_run_name,
    ) as run:
        operational_run_id = run.info.run_id

        mlflow.set_tags(
            {
                "evaluation.type": "aiq-operational",
                "evaluation.source_experiment_id": (
                    str(TRACE_EXPERIMENT_ID)
                ),
                "evaluation.source_dataset_id": (
                    str(DATASET_ID or "")
                ),
                "evaluation.source_trace_count": (
                    str(len(evaluation_data_df))
                ),
                "evaluation.expectation_checks_enabled": (
                    str(ENABLE_EXPECTATION_CHECKS).lower()
                ),
            }
        )

        operational_evaluation_result = evaluate(
            data=evaluation_data_df,
            scorers=evaluation_scorers,
        )

    print(
        "\nOperational trace evaluation completed."
    )
    print(
        "Operational evaluation run ID:",
        operational_run_id,
    )

    display_evaluation_result(
        operational_evaluation_result,
        "Operational",
    )

    verify_logged_run(
        operational_run_id,
        evaluation_experiment.experiment_id,
        "Operational",
    )

else:
    print("Operational evaluation is disabled.")


## 12. Read the evaluation in the MLflow UI

Open the `rh-research` experiment and select the run whose name begins with:

```text
aiq-operational-trace-evaluation-
```

The run provides two complementary views:

- **Run metrics:** aggregate values such as `web_search_count/mean` and `end_to_end_seconds/mean`
- **Linked trace assessments:** the individual operational values attached to each evaluated trace

With one trace, each `/mean` metric equals that trace's value. With a larger dataset, the run metrics summarize the complete evaluation set.

The evaluation run should show:

- `mlflow.runType = genai_evaluate`
- the source dataset and experiment tags
- the evaluated traces associated with the run
- the operational assessments created by `aiq_trace_operational_metrics`
- optional expectation assessments only when `ENABLE_EXPECTATION_CHECKS=True`


## 13. (Optional) Export an evaluation evidence bundle

This optional step creates a ZIP archive containing the dataset, trace summaries, full trace JSON, span inventory, grouped span summaries, and evaluation results.

Credential-like fields are redacted by key name, but prompts, responses, URLs, and custom text may still contain sensitive information. Review the archive before sharing it.


In [ ]:
import shutil
from datetime import datetime, timezone
from IPython.display import FileLink

SENSITIVE_KEY_PATTERN = re.compile(
    r"("
    r"authorization|"
    r"api[_-]?key|"
    r"access[_-]?token|"
    r"refresh[_-]?token|"
    r"bearer|"
    r"password|"
    r"secret|"
    r"cookie|"
    r"set-cookie"
    r")",
    re.IGNORECASE,
)


def sanitize_for_export(
    value: Any,
    key: Optional[str] = None,
) -> Any:
    if (
        key
        and SENSITIVE_KEY_PATTERN.search(str(key))
    ):
        return "<REDACTED>"

    if isinstance(value, dict):
        return {
            str(child_key): sanitize_for_export(
                child_value,
                str(child_key),
            )
            for child_key, child_value in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            sanitize_for_export(child)
            for child in value
        ]

    if isinstance(value, (str, int, float, bool)):
        return value

    if value is None:
        return None

    return str(value)


export_dir = Path(EXPORT_NAME)
if export_dir.exists():
    shutil.rmtree(export_dir)

export_dir.mkdir(parents=True)

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "mlflow_version": mlflow.__version__,
    "tracking_uri": mlflow.get_tracking_uri(),
    "workspace": MLFLOW_WORKSPACE,
    "trace_experiment_id": TRACE_EXPERIMENT_ID,
    "experiment_name": source_experiment.name,
    "experiment_id": str(TRACE_EXPERIMENT_ID),
    "expectation_checks_enabled": ENABLE_EXPECTATION_CHECKS,
    "dataset_id": DATASET_ID or None,
    "trace_filter": TRACE_FILTER,
    "trace_ids": trace_eval_df["trace_id"].tolist(),
    "include_trace_content": INCLUDE_TRACE_CONTENT_IN_EXPORT,
}

with open(
    export_dir / "manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sanitize_for_export(manifest),
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

trace_summary_df.to_csv(
    export_dir / "trace_summary.csv",
    index=False,
)

span_inventory_df.to_csv(
    export_dir / "span_inventory.csv",
    index=False,
)

if not span_inventory_df.empty:
    span_group_df.to_csv(
        export_dir / "span_type_name_summary.csv",
        index=False,
    )

if dataset is not None and dataset_df is not None:
    dataset_df.to_csv(
        export_dir / "evaluation_dataset.csv",
        index=False,
    )

    try:
        raw_dataset_export = dataset.to_dict()
    except Exception:
        raw_dataset_export = {
            "dataset_id": getattr(dataset, "dataset_id", DATASET_ID),
            "name": getattr(dataset, "name", None),
            "records": dataset_df.to_dict(orient="records"),
        }

    with open(
        export_dir / "evaluation_dataset.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            sanitize_for_export(raw_dataset_export),
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

if (
    operational_evaluation_result is not None
    and getattr(
        operational_evaluation_result,
        "result_df",
        None,
    )
    is not None
):
    operational_evaluation_result.result_df.to_csv(
        export_dir / "operational_evaluation_results.csv",
        index=False,
    )


for _, row in trace_eval_df.iterrows():
    trace = row["trace"]
    trace_id = row["trace_id"]

    try:
        trace_value = trace.to_dict()
    except Exception:
        trace_value = {
            "trace_id": trace_id,
            "serialization_error": (
                "trace.to_dict() was unavailable"
            ),
        }

    if not INCLUDE_TRACE_CONTENT_IN_EXPORT:
        trace_value = {
            "trace_id": trace_id,
            "info": sanitize_for_export(
                getattr(trace, "info", None)
            ),
            "span_summary": [
                {
                    "span_id": getattr(
                        span,
                        "span_id",
                        None,
                    ),
                    "parent_id": getattr(
                        span,
                        "parent_id",
                        None,
                    ),
                    "name": getattr(
                        span,
                        "name",
                        None,
                    ),
                    "span_type": span_type_text(span),
                    "status": span_status_text(span),
                    "duration_ms": span_duration_ms(span),
                    "model_name": get_span_model_name(span),
                    "attributes": sanitize_for_export(
                        span_attributes(span)
                    ),
                }
                for span in all_trace_spans(trace)
            ],
        }

    with open(
        export_dir / f"{trace_id}.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            sanitize_for_export(trace_value),
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

zip_path = shutil.make_archive(
    EXPORT_NAME,
    "zip",
    export_dir,
)

print("Created export:", zip_path)
display(FileLink(zip_path))

## What this demo demonstrates

This workflow shows how MLflow can evaluate an AI application from existing production-style traces:

- A UI-curated dataset defines a stable evaluation set.
- The historical execution is inspected rather than reproduced.
- AI-Q-specific span structure becomes deterministic operational assessments.
- The evaluation is logged as its own run in the same experiment and associated with the evaluated traces.
- The same pattern can expand to multiple traces for baseline comparison and regression detection.

A useful next step is to evaluate several comparable research traces, establish normal ranges or explicit operational requirements, and then enable the optional expectation checks in section 10.


## Current scope and limitations

This notebook intentionally focuses on **trace-based operational evaluation**.

It does not:

- rerun AI-Q or generate a new research response
- determine whether the final answer is factually correct
- treat latency, call count, search count, or parallelism as quality by themselves
- infer pass/fail thresholds automatically
- replace scenario-specific requirements, domain review, or human review

The result is a reproducible view of **how the research workflow executed**. The optional expectation section can turn selected operational measurements into explicit acceptance checks, but those checks are only meaningful when their thresholds come from real requirements or representative baselines.
